# Text-to-CAD Fine-tuning with CodeT5-base on Google Colab

**Model:** Salesforce/codet5-base (220M params) - optimized for code generation
**Data:** 100K+ synthetic CAD samples (18 parametric families)
**Runtime:** ~2-6 hours on Colab Pro (A100) / 12-24 hrs on Free (T4)
**Cost:** $0 (Free) or ~$10 (Colab Pro for A100)

## Features
- Train/val split with early stopping
- Cosine LR schedule with warmup
- BF16 on A100, FP16 on T4
- Gradient checkpointing for memory efficiency
- Metrics: exact match, valid JSON rate, valid operations rate
- Auto-resume from checkpoints

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {torch.cuda.get_device_capability()}")
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 8:
        print("✓ A100/H100 detected - will use BF16")
    else:
        print("✓ T4/V100 detected - will use FP16")

In [ ]:
# Clone the project repository
!git clone https://github.com/nagulapallidheeraj08-star/ai-cad-project.git 2>/dev/null || echo "Repo already exists"
%cd /content/ai-cad-project

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torch scikit-learn datasets tqdm

# Verify imports
import transformers
import torch
import sklearn
print(f"Transformers: {transformers.__version__}")
print(f"Torch: {torch.__version__}")
print(f"Sklearn: {sklearn.__version__}")

In [ ]:
# Generate enhanced synthetic training data (100K samples)
!python generate_enhanced_data.py --num-samples 100000 --output data/enhanced_cad_samples.json

In [ ]:
# Start fine-tuning - this will run for 2-24 hours depending on GPU
# Checkpoints saved every epoch, resumes automatically if interrupted
!python train_codet5_enhanced.py

In [ ]:
# Verify model was saved
import os

possible_paths = [
    "./cad_codet5_base_finetuned",
    "./ai-cad-project/cad_codet5_base_finetuned",
    "/content/cad_codet5_base_finetuned",
    "/content/ai-cad-project/cad_codet5_base_finetuned",
]

found = False
for path in possible_paths:
    if os.path.exists(path):
        print(f"\nFound model at: {path}")
        print(f"Contents: {os.listdir(path)}")
        for f in os.listdir(path):
            fp = os.path.join(path, f)
            print(f"  {f}: {os.path.getsize(fp)} bytes")
        found = True
        MODEL_DIR = path
        break
if not found:
    print("\nModel not found in expected locations")
    for root, dirs, files in os.walk("."):
        for f in files:
            if f in ["pytorch_model.bin", "model.safetensors", "config.json"]:
                print(f"  Found: {os.path.join(root, f)}")

In [ ]:
# Test the fine-tuned model
import json
import torch
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check if we have a valid MODEL_DIR
required_files = ["config.json", "model.safetensors"]
model_ready = False

if 'MODEL_DIR' in locals() and os.path.exists(MODEL_DIR):
    missing = [f for f in required_files if not os.path.exists(os.path.join(MODEL_DIR, f))]
    if not missing:
        model_ready = True
    else:
        print(f"Model dir exists but missing files: {missing}")

if not model_ready:
    print("Model not found or incomplete. Check 'Verify model' cell output.")
else:
    print(f"Loading model from: {MODEL_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)

    if torch.cuda.is_available():
        model = model.cuda()

    def generate_cad(text):
        inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,
                num_beams=4,
                early_stopping=True,
                temperature=0.7
            )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Test cases
    test_cases = [
        "Create a 50mm x 30mm x 20mm block with a 10mm hole",
        "Make a 40mm cube with 3mm fillets",
        "Design a cylinder 100mm long, 25mm diameter with 5mm keyway",
        "Create a hollow cylinder 30mm OD, 20mm ID, 50mm height",
        "Design a bracket 60x40x5mm with two 8mm holes 45mm apart and 30mm flange",
        "Create a conical frustum: bottom radius 20mm, top radius 10mm, height 30mm",
        "Make a stepped shaft: 30mm x 50mm, 20mm x 40mm, 15mm x 30mm",
        "Design a 100x80x10mm plate with 4x3 grid of 8mm holes spaced 25mm apart"
    ]

    print("=" * 80)
    for tc in test_cases:
        result = generate_cad(tc)
        print(f"Input:  {tc}")
        print(f"Output: {result}")
        try:
            spec = json.loads(result)
            print("  ✓ Valid JSON")
            if 'operations' in spec:
                print(f"  ✓ {len(spec['operations'])} operations")
        except:
            print("  ✗ Invalid JSON")
        print("-" * 80)

In [ ]:
# Create zip for download
import os
if 'MODEL_DIR' in locals() and os.path.exists(MODEL_DIR):
    zip_path = "/content/cad_codet5_base_finetuned.zip"
    print(f"Creating zip: {zip_path}")
    !zip -r "{zip_path}" "{MODEL_DIR}"
    print(f"Zip size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
    
    from google.colab import files
    files.download(zip_path)
else:
    print("Model not ready for download")

## Resume Training (if interrupted)

If Colab disconnects, re-run this notebook. The Trainer will auto-resume from the latest checkpoint in `cad_codet5_base_finetuned/`.

To manually resume:
```python
# In train_codet5_enhanced.py, add to TrainingArguments:
resume_from_checkpoint=True  # auto-resume from latest
# OR
resume_from_checkpoint="./cad_codet5_base_finetuned/checkpoint-XXXX"
```

## Next Steps After Training

1. **Download model**: `cad_codet5_base_finetuned.zip` from Colab
2. **Extract locally** to `C:\Users\nagul\Videos\AI CAD Project\cad_codet5_base_finetuned\`
3. **Update inference.py** to use the new model path
4. **Test locally** with your own text prompts

## For Even Better Results
- Increase samples to 500K+ (run generate_enhanced_data.py with --num-samples 500000)
- Train for 60+ epochs with larger model (CodeT5-large 770M)
- Add real CAD data from GitHub (DeepCAD, Fusion360 datasets)
- Use multi-GPU (Colab Pro+ or RunPod A100x4)